# 16 · Feature Map 시각화 — 계층적 특징 추출 증명 ⭐

**평가 항목 2번 (모델 설계, 40%) + 4번 (시각화, 15%) 동시 보강**

**목적**: ResNet-34 encoder의 **4-stage activation**을 시각화하여 **CNN의 계층적 특징 추출**을 시각적으로 증명.

**예상 패턴**:
- **Layer 1**: 저수준 — edge, 색상 패치
- **Layer 2**: 중간 — texture, 부분 패턴
- **Layer 3**: 고수준 — 얼굴 부위 (코, 눈 윤곽)
- **Layer 4**: 최고수준 — semantic abstraction

**산출물**: `outputs/feature_maps/feature_maps_all.png`

## 1. 환경 + 모델 로드

In [ ]:
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN').strip()
        result = subprocess.run(
            ['git', 'clone', f'https://{token}@github.com/keonU206/cv-final.git', REPO],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    !pip install -q segmentation_models_pytorch opencv-python

print(f'CWD: {os.getcwd()}')

In [ ]:
import torch
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
from project.segmentation.unet import build_unet

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_CKPT = '/content/drive/MyDrive/cv-final-ckpts/unet_attention_best.pt'

model = build_unet(
    num_classes=6,
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=3,
    attention_type='scse',
).to(device)

if os.path.exists(MODEL_CKPT):
    model.load_state_dict(torch.load(MODEL_CKPT, map_location=device))
    print(f'✅ Attention U-Net 로드')
else:
    print('⚠ ckpt 없음 — 초기 가중치 (시각화 의미 낮음)')

model.eval()
print(f'\nEncoder layers:')
for name in ['conv1', 'layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(model.encoder, name)
    print(f'  encoder.{name}: {type(layer).__name__}')

## 2. 사진 입력 + Forward Hook 등록

Hook으로 각 layer의 activation 추출 (계산 그래프 영향 X).

In [ ]:
SIZE = 256
OUTPUT_DIR = Path('outputs/feature_maps')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    print('얼굴 사진 업로드 (영문 파일명):')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    IMG_PATH = 'samples/original_before.png'

image_rgb = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))
input_tensor = torch.from_numpy(image_rgb).float() / 255.0
input_tensor = input_tensor.permute(2, 0, 1).unsqueeze(0).to(device)

# Forward hook 등록 — 5개 layer activation 저장
activations = {}
def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

hooks = []
for name in ['conv1', 'layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(model.encoder, name)
    h = layer.register_forward_hook(make_hook(name))
    hooks.append(h)

with torch.no_grad():
    _ = model(input_tensor)

for h in hooks:
    h.remove()

print(f'✅ Activations 추출 완료:')
for name, act in activations.items():
    print(f'  {name}: {tuple(act.shape)}')

## 3. Layer별 Feature Map 시각화 (8 채널씩 그리드)

각 layer의 첫 8개 채널을 그리드로 표시. 색상은 jet colormap.

In [ ]:
def viz_feature_map(act, name, n_channels=8):
    """act: (1, C, H, W) → 첫 n_channels grid 시각화."""
    feat = act[0]  # (C, H, W)
    n = min(n_channels, feat.shape[0])
    cols = 4
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes
    
    for i in range(n):
        fmap = feat[i].numpy()
        # 정규화
        fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
        axes[i].imshow(fmap, cmap='viridis')
        axes[i].set_title(f'ch {i}', fontsize=9)
        axes[i].axis('off')
    
    for i in range(n, len(axes)):
        axes[i].axis('off')
    
    fig.suptitle(f'{name} · shape {tuple(feat.shape)}', fontsize=13)
    plt.tight_layout()
    return fig

# 각 layer별로 PNG 저장
for name, act in activations.items():
    fig = viz_feature_map(act, name)
    save_path = OUTPUT_DIR / f'feature_{name}.png'
    fig.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✅ {save_path}')

## 4. 통합 비교 시각화 (Input + 4 Layer × 4 Channels)

In [ ]:
# 5 row × 5 col 그리드: Input + 4 layer × 4 channels (선택)
from matplotlib.gridspec import GridSpec

layer_names = ['conv1', 'layer1', 'layer2', 'layer3', 'layer4']
layer_titles = {
    'conv1': 'Conv1\n(stem, 64ch)',
    'layer1': 'Layer1\n(edges, 64ch)',
    'layer2': 'Layer2\n(texture, 128ch)',
    'layer3': 'Layer3\n(parts, 256ch)',
    'layer4': 'Layer4\n(semantic, 512ch)',
}

fig = plt.figure(figsize=(20, 14))
gs = GridSpec(5, 5, figure=fig, hspace=0.4, wspace=0.2)

# 첫 컬럼: 입력 이미지 (5 row 전체)
ax_input = fig.add_subplot(gs[:, 0])
ax_input.imshow(image_rgb)
ax_input.set_title('Input Image', fontsize=14, fontweight='bold')
ax_input.axis('off')

# 4 컬럼: layer별 4 channels
for row, name in enumerate(layer_names):
    act = activations[name][0]  # (C, H, W)
    for col in range(4):
        ax = fig.add_subplot(gs[row, col + 1])
        if col < act.shape[0]:
            fmap = act[col].numpy()
            fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
            ax.imshow(fmap, cmap='viridis')
            if col == 0:
                ax.set_ylabel(layer_titles[name], fontsize=11, fontweight='bold')
        ax.axis('off')
        if row == 0:
            ax.set_title(f'channel {col}', fontsize=10)

fig.suptitle('Feature Maps · CNN 계층적 특징 추출', fontsize=16, fontweight='bold', y=0.995)
save_all = OUTPUT_DIR / 'feature_maps_all.png'
fig.savefig(save_all, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 종합 시각화: {save_all}')

## 5. 채널 평균 시각화 (Layer별 "전체 추상화")

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 4))
axes[0].imshow(image_rgb)
axes[0].set_title('Input', fontsize=12)
axes[0].axis('off')

for i, name in enumerate(layer_names):
    act = activations[name][0]  # (C, H, W)
    mean_act = act.mean(dim=0).numpy()  # 채널 평균
    mean_act = (mean_act - mean_act.min()) / (mean_act.max() - mean_act.min() + 1e-8)
    axes[i+1].imshow(mean_act, cmap='viridis')
    axes[i+1].set_title(f'{name}\n{act.shape[0]}ch avg', fontsize=11)
    axes[i+1].axis('off')

fig.suptitle('Channel-Averaged Feature Maps · 계층적 추상화 흐름', fontsize=14, y=1.05)
plt.tight_layout()
save_avg = OUTPUT_DIR / 'feature_maps_averaged.png'
fig.savefig(save_avg, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 채널 평균 시각화: {save_avg}')

## 6. Drive 백업 + 학술 해석

In [ ]:
if IS_COLAB:
    DRIVE_OUT = Path('/content/drive/MyDrive/cv-final-ckpts/samples')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -v {OUTPUT_DIR}/*.png {DRIVE_OUT}/ 2>&1 | tail -10
    print(f'\n✅ Drive 백업: {DRIVE_OUT}/')

---

## 📚 학술 해석 (발표 슬라이드 멘트)

**관찰**:
- **Layer 1 (64ch)**: edge, 색상 패치 — 저수준 visual feature
- **Layer 2 (128ch)**: texture, 작은 부분 패턴 — 중간 수준
- **Layer 3 (256ch)**: 얼굴 부위 윤곽 (코, 눈, 입 위치) — 고수준
- **Layer 4 (512ch)**: semantic abstraction — 가장 추상적

**해상도 감소** (256 → 128 → 64 → 32 → 16 → 8):
- 공간 정보 ↓ but 의미 정보 ↑
- → "localization vs semantic" trade-off
- → U-Net의 **Skip Connection이 두 정보 모두 보존**하는 이유

**평가 항목 매핑**:
- **항목 2번 (모델 설계 40%)**: CNN의 계층적 특징 추출 **시각적 증명**
- **항목 4번 (시각화 15%)**: Feature Map 명시 (평가 기준에 직접 언급)

**참고문헌**:
- Zeiler, M. D., & Fergus, R. (2014). *Visualizing and Understanding Convolutional Networks*. ECCV.
- He, K. et al. (2016). *Deep Residual Learning for Image Recognition*. CVPR. (ResNet-34)
- Ronneberger, O. et al. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI.

**발표 멘트 예시**:
> "본 시각화는 **CNN의 계층적 특징 추출**을 직접 보여줍니다 (Zeiler 2014 ECCV).
> Layer 1은 edge·색상 같은 저수준 feature를, Layer 4는 얼굴 부위에 대한 high-level semantic을 학습합니다.
> 해상도는 256→8픽셀로 32배 감소하지만 채널은 64→512로 8배 증가합니다 — 
> **공간 정보를 의미 정보로 변환**하는 과정입니다.
> U-Net의 **Skip Connection**은 이 두 정보를 모두 보존하여 
> 정확한 localization + semantic understanding을 동시에 달성합니다."